# vEMDestripe: Bumbarger13 comparison
### Original-source EMISAC edition / 官方原版结果演示

Compare **vEMDestripe, EMISAC, FECNet and SSCOR** on 32 aligned 1024×1024
Bumbarger13 sections (local Z 777–808). This is a small demonstration,
**not the paper's full-volume benchmark**. There is no paired clean target.

**EMISAC:** results generated by the unchanged official Python 2 source
in a recorded legacy environment. The notebook loads these precomputed
results; it does not pretend to run the original algorithm live in Python 3.
Original source, raw floating-point H5, parameters, logs and checksums are included.

**vEMDestripe:** live released-checkpoint inference on GPU, or an explicitly
labelled precomputed fallback. FECNet and SSCOR use documented frozen
comparison outputs; their adaptation/training provenance is shown below.

展示默认使用 **Raw outputs（原始输出）**。整体亮度匹配只作为明确标注的可选视图，
不会覆盖原始结果。点击菜单 **Runtime → Run all / 运行时 → 全部运行**。

## 1. Connect and configure / 连接与设置

A GPU is optional. Select a T4 GPU to run vEMDestripe live; a CPU session
still completes the comparison using the released-checkpoint fallback.
No GitHub account, token, training data or third-party checkpoint upload is needed.
The first run downloads the versioned demo bundle and installs dependencies.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

GITHUB_OWNER, REPOSITORY = "simple-happy-802", "vEMDestripe"
REPOSITORY_REF = "demo-official-v2"
RELEASE_TAG = "demo-official-v2"
ASSET_NAME = "vemdestripe_bumbarger_official_demo_v2.zip"
ASSET_SHA256 = "9d4ec82b7ebc6fcbcabf28fca1133427e2912feffc9018783c7c3dc94c688ea7"
ASSET_URL = (f"https://github.com/{GITHUB_OWNER}/{REPOSITORY}/releases/"
             f"download/{RELEASE_TAG}/{ASSET_NAME}")
RUN_VEMDESTRIPE_LIVE = True
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
RUNTIME_ROOT = Path('/content') if IN_COLAB else Path.cwd()/'official_demo_runtime'
PROJECT_ROOT = RUNTIME_ROOT/REPOSITORY
DOWNLOAD_ROOT, RESULT_ROOT = RUNTIME_ROOT/'downloads', RUNTIME_ROOT/'official_comparison_results'
for folder in (RUNTIME_ROOT, DOWNLOAD_ROOT, RESULT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

def run_command(command, **kwargs):
    result = subprocess.run([str(x) for x in command], text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)
    if result.returncode:
        print(result.stdout[-12000:])
        raise RuntimeError(f'Command failed (exit {result.returncode})')
    return result.stdout
print(f'Runtime: {RUNTIME_ROOT}; Colab: {IN_COLAB}')

In [ ]:
repo_url = f'https://github.com/{GITHUB_OWNER}/{REPOSITORY}.git'
if not (PROJECT_ROOT/'.git').is_dir():
    run_command(['git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF,
                 repo_url, PROJECT_ROOT])
commit = run_command(['git', '-C', PROJECT_ROOT, 'rev-parse', 'HEAD']).strip()
tagged_commit = run_command(['git', '-C', PROJECT_ROOT, 'rev-parse',
                             REPOSITORY_REF+'^{commit}']).strip()
if commit != tagged_commit:
    raise RuntimeError('Existing checkout is not at the versioned demo tag. Use a fresh runtime.')
if run_command(['git', '-C', PROJECT_ROOT, 'status', '--porcelain',
                '--untracked-files=no']).strip():
    raise RuntimeError('The demo source checkout has modified tracked files; use a fresh runtime.')
print('Installing demo dependencies; the existing PyTorch build is retained if compatible.')
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-e',
             f'{PROJECT_ROOT}[evaluation]', 'pandas>=2.2,<3',
             'ipywidgets>=8,<9', 'requests>=2.32,<3', 'h5py>=3.11,<4'])
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT/'colab'))
print(f'Versioned project source: {REPOSITORY_REF}, commit {commit}')

## 2. Download and verify / 下载与校验

The archive SHA256 is pinned in this notebook. Each bundled file is checked
again against the manifest. A failed check stops execution rather than
silently substituting a different EMISAC implementation.

In [ ]:
import requests
from official_demo import sha256_file, safe_extract_bundle, verify_bundle

archive = DOWNLOAD_ROOT/ASSET_NAME
if not archive.is_file():
    partial = archive.with_suffix('.zip.part')
    print('Downloading the original-source demo bundle…')
    with requests.get(ASSET_URL, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        with partial.open('wb') as stream:
            for chunk in response.iter_content(1024*1024):
                if chunk:
                    stream.write(chunk)
    if sha256_file(partial) != ASSET_SHA256:
        raise RuntimeError('Downloaded archive checksum mismatch; do not use these results.')
    partial.replace(archive)
if sha256_file(archive) != ASSET_SHA256:
    raise RuntimeError('Cached archive checksum mismatch. Start with a fresh runtime.')
ASSET_ROOT = safe_extract_bundle(archive, DOWNLOAD_ROOT/'official_bundle', ASSET_SHA256)
manifest = verify_bundle(ASSET_ROOT)
print(f'Verified archive: {archive.stat().st_size/1024**2:.1f} MiB')
print('Every bundled file passed its SHA256 check.')

## 3. What exactly generated EMISAC? / 原版结果的来源

The official commit is fixed. Z upsampling was explicitly disabled in the
original run (`up_fg=0`), following an option shown in the upstream README;
this is **not a claim to use all original defaults**. The input to EMISAC
was exactly this 32-section window. Its old dependency environment is
documented, but is not claimed to recover every detail of the authors' historical machine.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, HTML, Code, FileLink
from official_demo import official_metadata, verify_emisac_pixels

emisac_metadata = official_metadata(ASSET_ROOT)
display(emisac_metadata)
display(verify_emisac_pixels(ASSET_ROOT))
provenance_root = ASSET_ROOT/'provenance'/'emisac_official'
display(Markdown(
    '**EMISAC mode: original-source, legacy-environment precomputed output.** '
    'No Python 3 compatibility result is used in this edition.'))
display(Markdown(f'[Official source and reproducibility instructions]('
                 f'https://github.com/{GITHUB_OWNER}/{REPOSITORY}/blob/{commit}/'
                 'colab/official_emisac/REPRODUCE.md)'))

## 4. vEMDestripe inference / 本方法推理

The released checkpoint uses mean reference fusion and **16×16 block
averaging**. All 11 supplied, independently screened reference sections
are used (`num_reference=16` is an upper limit). At 1024×1024, sliding-window
blending is disabled. No per-slice percentile normalization or Z postprocessing is added.

With no GPU, the notebook explicitly selects the bundled released-checkpoint
output. A live inference error is not silently replaced with a fallback.

In [ ]:
import torch
checkpoint = PROJECT_ROOT/'checkpoints'/'vemdestripe_seed42'/'best.pth'
CHECKPOINT_SHA256 = '9a01cdef1f09716fe87369b057633f9e1278e86e77cbc70eb66b75798709c44a'
if sha256_file(checkpoint) != CHECKPOINT_SHA256:
    raise RuntimeError('Released checkpoint checksum mismatch.')
LIVE_VEM_DIR = None
if RUN_VEMDESTRIPE_LIVE and torch.cuda.is_available():
    LIVE_VEM_DIR = RESULT_ROOT/'vEMDestripe_live'
    LIVE_VEM_DIR.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, PROJECT_ROOT/'infer.py',
        '--input_dir', ASSET_ROOT/'input', '--output_dir', LIVE_VEM_DIR,
        '--model', checkpoint, '--reference', ASSET_ROOT/'reference',
        '--num_reference', '16', '--reference_mode', 'fused',
        '--reference_preprocess', 'block_mean', '--struct_block_size', '16',
        '--reference_selection', 'uniform', '--reference_seed', '42',
        '--image_size', '1024', '--inference_size', '1024',
        '--no_sliding_window', '--no_z_context', '--no_z_postprocess',
        '--no_preview', '--disable_progress_bar']
    started = time.time()
    log = run_command(command)
    (RESULT_ROOT/'vemdestripe_inference.log').write_text(log, encoding='utf-8')
    VEM_SOURCE = 'live released-checkpoint GPU inference'
    print(f'Live inference finished in {(time.time()-started)/60:.1f} min')
else:
    VEM_SOURCE = 'precomputed released-checkpoint output (no GPU or live mode disabled)'
    print(VEM_SOURCE)
print('vEMDestripe source:', VEM_SOURCE)

In [ ]:
from official_demo import load_official_demo
from demo_metrics import METHOD_ORDER
volumes, section_ids = load_official_demo(ASSET_ROOT, live_vem_dir=LIVE_VEM_DIR)
sources = {
    'Original': 'unaltered aligned input',
    'EMISAC': 'unchanged official source + fixed legacy environment; precomputed on these 32 sections',
    'FECNet': 'frozen paper-comparison grayscale adaptation / Bumbarger-specific fine-tuning',
    'SSCOR': 'frozen dataset-specific seed-42 paper-comparison run',
    'vEMDestripe': VEM_SOURCE}
display(pd.DataFrame([{'method': name, 'shape_ZYX': str(volumes[name].shape),
                       'result_source': sources[name]} for name in METHOD_ORDER]))
print('Aligned local sections:', section_ids[0], 'through', section_ids[-1])

## 5. Inspect XY, XZ and YZ / 交互查看

**Raw outputs is the default.** All methods share the same 0–255 grayscale
display range. The optional matched view applies one scale and bias to each
complete 32-section volume, not a separate adjustment per slice.
Cross-plane views explicitly enlarge Z by 6× for visibility; native-Z
continuity statistics are calculated before any such interpolation.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact, IntSlider
from demo_metrics import matched_volumes, plot_xy, plot_cross_sections, plot_z_profiles
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
matched, affine_transforms = matched_volumes(volumes)
display_mode = widgets.ToggleButtons(options=['Raw outputs', 'Global affine matched'],
                                    value='Raw outputs', description='Display:')
display(display_mode)

@interact(z=IntSlider(min=0, max=31, value=16, description='Z offset'), mode=display_mode)
def inspect_xy(z, mode):
    print(mode)
    plot_xy(volumes if mode == 'Raw outputs' else matched, section_ids, z)

In [ ]:
@interact(y=IntSlider(min=32, max=991, step=8, value=512, description='Y'), mode=display_mode)
def inspect_xz(y, mode):
    print(mode)
    plot_cross_sections(volumes if mode == 'Raw outputs' else matched,
                        axis='xz', position=y, z_ratio=6.0)

@interact(x=IntSlider(min=32, max=991, step=8, value=512, description='X'), mode=display_mode)
def inspect_yz(x, mode):
    print(mode)
    plot_cross_sections(volumes if mode == 'Raw outputs' else matched,
                        axis='yz', position=x, z_ratio=6.0)

In [ ]:
@interact(mode=display_mode)
def inspect_brightness(mode):
    print(mode)
    plot_z_profiles(volumes if mode == 'Raw outputs' else matched, section_ids)

## 6. Recompute diagnostics / 现场计算指标

Both **raw** and **global-affine-matched** results are reported separately.
ASID_MAD is the MAD of adjacent whole-section mean differences, not a
pixelwise ASID percentage. Local-ASID, Z-HF and domain-calibrated NIQE are
diagnostics; no single number proves preservation of biological structure.
Contrast and blur are not a standalone quality ranking.

NIQE uses unpaired screened references, not a clean target for these slices.
No PSNR or SSIM is claimed. These 32-section diagnostics must not replace
the paper's full evaluation. The calculation can take several minutes.

In [ ]:
from official_demo import evaluate_outputs
niqe_model = ASSET_ROOT/'bumbarger13_niqe_model.npz'
raw_metrics = evaluate_outputs(volumes, niqe_model, mode='raw',
                               xy_samples=8, cross_sections=6, z_ratio=6.0, border=32)
print('RAW OUTPUTS — no additional brightness matching')
display(raw_metrics.round(4))
raw_metrics.to_csv(RESULT_ROOT/'metrics_raw.csv')

In [ ]:
matched_metrics = evaluate_outputs(volumes, niqe_model, mode='global_affine_matched',
                                   xy_samples=8, cross_sections=6, z_ratio=6.0, border=32)
print('GLOBAL AFFINE MATCHED — one transform per complete volume')
display(matched_metrics.round(4))
matched_metrics.to_csv(RESULT_ROOT/'metrics_global_affine_matched.csv')
print('Both CSV files retain their explicit processing-mode labels.')

## 7. Inspect the source / 查看源码

EMISAC is read directly from the verified bundle: these are the exact two
unchanged files used to generate its result. vEMDestripe is read from this
versioned checkout. FECNet/SSCOR links show their pinned upstream source;
their frozen comparison outputs have the adaptation/training provenance
listed above and are not claimed to come from universal default weights.

In [ ]:
from functools import lru_cache
source_specs = {
    'vEMDestripe': {'repo': f'{GITHUB_OWNER}/{REPOSITORY}', 'commit': commit,
        'paths': ['vemdestripe/model.py', 'infer.py', 'vemdestripe/reference.py', 'vemdestripe/loss.py']},
    'EMISAC': {'repo': 'azadis/EMISAC',
        'commit': '7a82a17b7bbb4ca14dff2d1c1a4a018fc8cebaf5',
        'paths': ['EM_normalization.py', 'normalize_partition.py']},
    'FECNet': {'repo': 'KevinJ-Huang/FECNet',
        'commit': 'fc5a9eb224c448d190919558cc87ea7b7447f7f4',
        'paths': ['FastFourierSICE/models/archs/EnhanceN_arch.py', 'FastFourierSICE/test.py']},
    'SSCOR': {'repo': 'AI-XPM-group/SSCOR',
        'commit': '985479cd79bcf1359e3d9ba44bacd5f372eb2e60',
        'paths': ['models/networks.py', 'restore.py', 'train.py']}}

@lru_cache(maxsize=20)
def fetch_source(method, file):
    if method == 'vEMDestripe':
        return (PROJECT_ROOT/file).read_text(encoding='utf-8')
    if method == 'EMISAC':
        return (provenance_root/'source'/file).read_text(encoding='utf-8')
    spec = source_specs[method]
    url = f"https://raw.githubusercontent.com/{spec['repo']}/{spec['commit']}/{file}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text

method_choice = widgets.Dropdown(options=list(source_specs), description='Method:')
path_choice = widgets.Dropdown(options=source_specs['vEMDestripe']['paths'],
                              description='File:', layout=widgets.Layout(width='650px'))
show_all = widgets.Checkbox(value=False, description='Show complete file')
source_output = widgets.Output()
def render_source(change=None):
    with source_output:
        source_output.clear_output(wait=True)
        method, file = method_choice.value, path_choice.value
        if file not in source_specs[method]['paths']:
            return
        spec = source_specs[method]
        url = f"https://github.com/{spec['repo']}/blob/{spec['commit']}/{file}"
        display(Markdown(f'**{method}** — [complete pinned source]({url})'))
        try:
            lines = fetch_source(method, file).splitlines()
            selected = lines if show_all.value else lines[:200]
            display(Code(chr(10).join(selected), language='python'))
            print(f'Showing {len(selected)}/{len(lines)} lines. Result source: {sources[method]}')
        except Exception as error:
            print(f'Inline source unavailable: {error}. Use the pinned source link above.')
def change_method(change):
    path_choice.options = source_specs[change['new']]['paths']
    render_source()
method_choice.observe(change_method, names='value')
path_choice.observe(render_source, names='value')
show_all.observe(render_source, names='value')
display(widgets.VBox([method_choice, path_choice, show_all, source_output]))
render_source()

## 8. Export and independently reproduce / 下载与独立复算

The export contains the raw displayed method PNGs, the original EMISAC H5
and provenance, separate raw/matched CSV tables, and this session's source
labels. Precomputed results remain labelled as precomputed.

Optional check: switch between raw and matched views, then run the next cell.
The file-integrity check must still pass: changing the view does not alter
the official raw result. To actually rerun original EMISAC, use the versioned
Docker instructions linked above outside the ordinary hosted Colab runtime.

In [ ]:
verify_bundle(ASSET_ROOT)
print('Self-check passed: original bundle files remain unchanged after display and evaluation.')
runtime_provenance = {
    'repository': repo_url, 'version_tag': REPOSITORY_REF, 'project_commit': commit,
    'asset_sha256': ASSET_SHA256, 'local_z_half_open': [777, 809],
    'result_sources': sources, 'vEMDestripe_checkpoint_sha256': CHECKPOINT_SHA256,
    'EMISAC': emisac_metadata,
    'metric_modes': ['raw', 'global_affine_matched'],
    'no_paired_clean_target': True,
    'note': '32-section demo diagnostics, not full-paper benchmark results'}
(RESULT_ROOT/'runtime_provenance.json').write_text(
    json.dumps(runtime_provenance, indent=2, ensure_ascii=False), encoding='utf-8')
from PIL import Image
for method, volume in volumes.items():
    folder = RESULT_ROOT/'raw_outputs'/method
    folder.mkdir(parents=True, exist_ok=True)
    for section_id, image in zip(section_ids, volume):
        Image.fromarray(image).save(folder/f'slice_{section_id:05d}.png')
shutil.copytree(provenance_root, RESULT_ROOT/'emisac_official_provenance', dirs_exist_ok=True)
shutil.copy2(ASSET_ROOT/'manifest.json', RESULT_ROOT/'bundle_manifest.json')
for source in (ASSET_ROOT/'DATA_LICENSE.txt',
               PROJECT_ROOT/'colab'/'DATA_LICENSE.md',
               PROJECT_ROOT/'colab'/'THIRD_PARTY_METHODS.md'):
    if source.is_file():
        shutil.copy2(source, RESULT_ROOT/source.name)
export_zip = Path(shutil.make_archive(str(RUNTIME_ROOT/'vemdestripe_official_demo_results'),
                                    'zip', RESULT_ROOT))
print(f'Export: {export_zip.name} ({export_zip.stat().st_size/1024**2:.1f} MiB)')
if IN_COLAB:
    from google.colab import files
    button = widgets.Button(description='Download results ZIP', icon='download')
    button.on_click(lambda _: files.download(str(export_zip)))
    display(button)
else:
    display(FileLink(str(export_zip)))

### What this demonstration establishes

It exposes the exact recorded EMISAC original-source result and lets you
inspect its data and code, compare aligned images, and recompute transparent
diagnostics. It does not claim that the Python 3 migration is numerically
equivalent, or that a 32-section comparison reproduces the full paper.

使用方法：分享本 Notebook 的 Open in Colab 链接。读者连接运行时后选择全部运行，
即可查看各方法来源、原始结果、可选匹配视图、两套指标及源码，并下载结果包。